# 單元 14.9 APCS 實作真題特訓（初級題）：g595. 修補圍籬【48 Cells 極致緩坡超詳解版】

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**對應教材**：吳邦一老師《Python 程式設計從 APCS 實作 1 級到 3 級》第 11 章（第 42–43 頁）  
**題型定位**：APCS 實作真題特訓——初級題（對應舊版實作第 1 題，APCS 2021 年 11 月場次，ZeroJudge g595）  

---

### 🗺️ 本單元學習地圖與通關導航
在前面的單元中，我們征服了三邊長判斷、大數差值、邏輯運算子、比分統計、購物車抵銷、人力分配二次函數、購買力全距以及七言對聯格律。
現在，我們來到陣列演算法中最具啟發性的經典實戰大關——**g595. 修補圍籬（Repair Fence）**！

這道題目模擬農場生活中的實用修繕問題：
強烈颱風過後，農場一整排由 $n$ 根木樁組成的圍籬中，有些木樁被吹斷倒塌了（高度變成了 `0`）。
為了節省修繕材料並兼顧防護效果，農場主人決定：**將每一處損壞的木樁，修補到其「左右相鄰兩根木樁中較矮的那一根」的高度**。
修補每根木樁的費用等於其修補後的高度，題目要求我們計算出修補整排圍籬所需的**總花費**。

這道題目表面上看似只是簡單的 `min()` 取較小值，但它暗藏了所有初學程式設計者在陣列操作時最容易崩潰的**「兩端邊界存取陷阱（Boundary Edge Cases）」**：
如果直接寫 `h[i-1]` 或 `h[i+1]`，在最左端會發生索引負數倒轉（拿到最後一根木樁），在最右端更會直接觸發 `IndexError: list index out of range` 當場崩潰！

為了讓初學者零負擔徹底搞懂陣列相鄰探測與邊界防禦，本單元特別升級為**「8 大微階梯極致緩坡架構」（全篇 48 儲存格）**：
1. **14.9.1 🧱 圍籬損壞修復模型與生活幾何剖析**：建立一維木樁高低起伏與損壞高度為 0 的直觀幾何感知。
2. **14.9.2 🔍 一維陣列相鄰探測與損壞定位（`h[i] == 0`）**：掌握串列走訪掃描，利用題目「絕無連續損壞」之保證建立信心。
3. **14.9.3 ⚠️ 內部一般破損修復：雙側相鄰取較小值（`min(h[i-1], h[i+1])`）**：推導中間位置修復成本最小化公式。
4. **14.9.4 🚫 邊界越界大陷阱：最左端（`i == 0`）與最右端（`i == n-1`）**：剖析 `IndexError` 與負索引倒轉致命盲區，建立邊界特判意識。
5. **14.9.5 🥋 傳統邊界特判解法：三分支互斥防禦與總花費累加**：以結構化 `if-elif-else` 實作最扎實的考場第一道防線。
6. **14.9.6 ✨ 陣列降維神技：哨兵前後加框法（Sentinel Technique）**：解鎖高階競賽選手絕技，在前後補上「虛擬無敵木樁」，一行代碼消滅所有邊界特判！
7. **14.9.7 🏆 完整 AC 模組拼裝與雙解法（傳統特判 vs 哨兵加框）對照**：手把手組裝滿分程式碼，對比兩種思維的優劣與美感。
8. **14.9.8 🧪 極端邊界壓力測試、無損壞檢驗與考場常見失分地雷排查**：覆蓋兩端破損、全無破損、極小尺寸等邊界測試，完美通關！

帶上你的工具箱，讓我們一起修好這排圍籬，掌握陣列邊界處理的最高心法！

## 📜 【APCS 官方完整真題題面與規範】g595. 修補圍籬

> 📌 **題目資訊快覽**  
> * **題目名稱**：修補圍籬 (Repair Fence)  
> * **題目出處**：APCS 大學程式設計先修檢測（2021 年 11 月場次）實作題第 1 題（初級題）  
> * **線上評判**：ZeroJudge g595 / 高中生程式解題系統  
> * **對應講義**：吳邦一老師《Python 程式設計從 APCS 實作 1 級到 3 級》第 11 章（第 42–43 頁）  
> * **難度評級**：★☆☆☆☆（初級題 / 一維陣列相鄰探測、邊界條件防禦、極值運算、哨兵加框）  

---

### 📝 題目描述（Problem Description）
農場裡有一排由 $n$ 根木樁組成的圍籬，由左至右依序編號為 $0$ 到 $n - 1$。
每根木樁的高度記錄在一個整數序列 $h_0, h_1, \dots, h_{n-1}$ 中。
經歷了一場強烈颱風後，有些木樁被吹斷了，其高度變成了 **`0`**。

```text
   【農場圍籬破損圖解】
   高度
    8  │       █
    6  │   █   █       █
    4  │   █   █   ░   █       ░ = 損壞木樁 (高度 0)
    2  │   █   █   ░   █       █ = 完好木樁
    0  └───┴───┴───┴───┴───
   木樁:   0   1   2   3
   高度:   6   8   0   6
```

農場主人決定修補這些損壞的木樁，修復規則如下：
1. **中間損壞木樁**：若位置 $i$（$0 < i < n - 1$）的木樁高度為 0，其修補後的**高度為其「左邊鄰居」與「右邊鄰居」中較小的那一個高度**，即：  
   $$\text{Cost} = \min(h_{i-1}, h_{i+1})$$
2. **最左端損壞木樁**：若最左端位置 $0$ 的木樁高度為 0，因為它沒有左邊鄰居，所以其修補後的**高度直接等於其右邊鄰居的高度**，即：  
   $$\text{Cost} = h_1$$
3. **最右端損壞木樁**：若最右端位置 $n - 1$ 的木樁高度為 0，因為它沒有右邊鄰居，所以其修補後的**高度直接等於其左邊鄰居的高度**，即：  
   $$\text{Cost} = h_{n-2}$$

⚠️ **重要題目保證**：
題目保證**「不會有連續兩根相鄰的木樁同時損壞」**（即序列中絕對不會出現連續兩個 0）。這意味著任何損壞木樁的相鄰鄰居，其高度必定大於 0！

修補每根損壞木樁的費用等於其修補後的高度。請計算並輸出**修補所有損壞木樁所需的總費用**。

---

### 📥 輸入格式（Input Format）
- 第 1 行：包含一個正整數 $n$（$2 \le n \le 100$），代表圍籬木樁的總數量。
- 第 2 行：包含 $n$ 個非負整數 $h_0, h_1, \dots, h_{n-1}$（$0 \le h_i \le 100$），代表各木樁的高度，數值間以半形空格隔開。

### 📤 輸出格式（Output Format）
- 輸出僅有一行，包含一個非負整數，代表**修補所有木樁的總花費**。
- 若圍籬完全沒有損壞（無任何 0），請輸出 `0`。

---

### 🧪 官方範例測資一覽表

| 範例編號 | 輸入範例 | 正確輸出 | 詳細修復過程推導 |
| :--- | :--- | :--- | :--- |
| **範例一** | `5`<br>`3 0 5 0 4` | `7` | 位置 1 損壞（中間）：相鄰高度為 3 與 5，取 $\min(3, 5) = 3$。<br>位置 3 損壞（中間）：相鄰高度為 5 與 4，取 $\min(5, 4) = 4$。<br>總花費為 $3 + 4 = 7$。 |
| **範例二** | `4`<br>`0 6 8 0` | `14` | 位置 0 損壞（最左端）：無左鄰居，直接取右鄰居高度 $h_1 = 6$。<br>位置 3 損壞（最右端）：無右鄰居，直接取左鄰居高度 $h_2 = 8$。<br>總花費為 $6 + 8 = 14$。 |
| **範例三** | `6`<br>`10 20 0 30 0 15` | `35` | 位置 2 損壞：取 $\min(20, 30) = 20$。<br>位置 4 損壞：取 $\min(30, 15) = 15$。<br>總花費為 $20 + 15 = 35$。 |
| **範例四** | `3`<br>`5 8 6` | `0` | 全體木樁皆完好無損（無任何 0），不需要任何修繕，輸出 `0`。 |

---

### ⚖️ 評分說明與測資限制
- 執行時間限制：1.0 秒，記憶體限制：64 MB。
- $2 \le n \le 100$，$0 \le h_i \le 100$。
- 題目保證不出現連續相鄰的 0。

### 14.9.1 🧱 圍籬損壞修復模型與生活幾何剖析

在開始動手寫程式前，我們先用直觀的生活比喻來理解這道題目的精神。
想像你是一位農場工人，颱風天過後巡視農場邊界：
一整排木樁原本整整齊齊，但有幾根被風吹斷，倒在地上成了平地（高度為 0）。農場主人交代你：「拿木料來補，把斷掉的木樁補起來！」

#### 為什麼要補到「左右相鄰較矮的那一根」？
這是一個兼顧**成本控制**與**外觀平整**的工程決策：
- 如果左邊木樁高 3 公尺，右邊木樁高 5 公尺；
- 若補到 5 公尺，會浪費更多木料，而且跟左邊 3 公尺比起來顯得突兀；
- 若補到 3 公尺，木料最省，又能完美跟左邊連成平滑的防護線！
因此，在數學上修補的高度就是：$\min(\text{左邊高度}, \text{右邊高度})$。

```text
     左鄰居(3)      損壞(0)       右鄰居(5)
       [3]           [?]           [5]
       [3]        修復為 min       [5]
       [3]    ───────────────>    [5]
       [3]           [3]           [5]
       [3]           [3]           [5]
    ======================================== (地面)
```

在 Python 中，兩數取較小值的內建函數是 **`min(a, b)`**。計算每根木樁的修補成本，本質上就是一次簡單的 `min` 運算！

In [ ]:
# ==========================================
# [2] Code 範例區：使用 min() 計算單處木樁修補高度
# ==========================================

left_h = 3
right_h = 5

# 中間破損木樁修補高度為左右較小值
repaired_h = min(left_h, right_h)

print("=== 單處修補成本範例 ===")
print(f"左鄰居高度: {left_h}，右鄰居高度: {right_h}")
print(f"修補後高度（等於修繕花費）: {repaired_h}")


In [ ]:
# ==========================================
# [3] Code 填空題：補齊 min() 運算求修復高度
# 任務說明：請將下方 ___ 替換為 min 函數，計算出破損處的修復高度。
# ==========================================

left = 20
right = 15

# 取兩者較小值
cost = ___(left, right)

print(f"修補花費為: {cost}")
# 預期輸出：修補花費為: 15


In [ ]:
# ==========================================
# [4] Code 練習題：計算多處獨立破損的修補成本清單
#
# 【公開測試資料 1】
# 相鄰高度對：(3, 5) 與 (5, 4)
# 預期輸出：[3, 4]
#
# 【公開測試資料 2】
# 相鄰高度對：(20, 30) 與 (30, 15)
# 預期輸出：[20, 15]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def calc_costs(neighbor_pairs):
    return [min(l, r) for l, r in neighbor_pairs]

# 執行測試
print("測試 1:", calc_costs([(3, 5), (5, 4)]))
print("測試 2:", calc_costs([(20, 30), (30, 15)]))


In [ ]:
# ==========================================
# [5] Code 挑戰題：若修補規則改為「取左右較大值」
# 任務說明：如果農場主人要求以最高標準防護，損壞處要修補到左右較高的高度。
#          請寫一個小函式，計算左右鄰居 (l, r) 的修補成本。
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


### 14.9.2 🔍 一維陣列相鄰探測與損壞定位（`h[i] == 0`）

整排圍籬有 $n$ 根木樁，記錄在串列 `h` 中，索引由 `0` 到 `n - 1`。
要修補圍籬，我們必須先派出偵察兵，**循序掃描每一根木樁，找出哪些位置的高度是 0**！

#### 如何遍歷串列的每一個位置？
在 Python 中，最標準也最強大的方式是使用 `range(n)` 迴圈：
```python
for i in range(n):
    if h[i] == 0:
        # 抓到了！第 i 根木樁斷掉了！
```

#### 🌟 題目給定的一顆「超級定心丸」：
在題目說明中有一行極其關鍵的保證：
> ⚠️ **「題目保證不會有連續兩根相鄰的木樁同時損壞」**

這句話對解題有什麼重大意義？
1. 如果允許連續兩個 0（例如 `[3, 0, 0, 5]`），那麼當你在算第一個 0 時，它的右邊是 0，還沒修好，根本拿不出有效高度！這會演變成複雜的連續填補或遞迴演算法。
2. 但因為**保證不連續損壞**，意味著：**任何一個高度為 0 的木樁，它的左右鄰居必定都是完好完好的正整數（> 0）**！
這讓我們的掃描過程變得非常純粹：看到 0，直接看旁邊的木樁，絕對不會看到另一個 0！

In [ ]:
# ==========================================
# [2] Code 範例區：掃描並印出所有損壞木樁的位置索引
# ==========================================

h = [3, 0, 5, 0, 4]
n = len(h)

broken_indices = []
for i in range(n):
    if h[i] == 0:
        broken_indices.append(i)

print("=== 損壞木樁掃描結果 ===")
print(f"圍籬高度序列: {h}")
print(f"損壞位置索引: {broken_indices}（共有 {len(broken_indices)} 處破損）")


In [ ]:
# ==========================================
# [3] Code 填空題：計數高度為 0 的木樁總數
# 任務說明：請將下方 ___ 替換，統計序列中有幾根木樁高度為 0。
# ==========================================

fence = [10, 20, 0, 30, 0, 15]

# 使用 count 統計 0 的個數
zero_count = fence.count(___)

print(f"破損木樁總數: {zero_count}")
# 預期輸出：破損木樁總數: 2


In [ ]:
# ==========================================
# [4] Code 練習題：找出所有損壞處的前後鄰居高度
#
# 【公開測試資料 1】
# 輸入：h = [3, 0, 5, 0, 4]
# 預期輸出：位置 1 的鄰居為 (3, 5)，位置 3 的鄰居為 (5, 4)
#
# 【公開測試資料 2】
# 輸入：h = [10, 20, 0, 30, 0, 15]
# 預期輸出：位置 2 的鄰居為 (20, 30)，位置 4 的鄰居為 (30, 15)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def find_inner_neighbors(h):
    res = []
    for i in range(1, len(h) - 1):
        if h[i] == 0:
            res.append(f"位置 {i} 的鄰居為 ({h[i-1]}, {h[i+1]})")
    return "，".join(res)

# 執行測試
print("測試 1:", find_inner_neighbors([3, 0, 5, 0, 4]))
print("測試 2:", find_inner_neighbors([10, 20, 0, 30, 0, 15]))


In [ ]:
# ==========================================
# [5] Code 挑戰題：驗證序列是否符合「無連續損壞」保證
# 任務說明：請寫一個函式，檢查給定序列中是否真的「沒有任何連續兩個 0」。
#          若完全無連續 0 則回傳 True，若出現連續 0 則回傳 False。
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


### 14.9.3 ⚠️ 內部一般破損修復：雙側相鄰取較小值（`min(h[i-1], h[i+1])`）

現在我們聚焦於整排木樁的「內部區域」，也就是排除最左端與最右端之後的中間木樁。

#### 中間位置的索引範圍：
若總共有 $n$ 根木樁（索引 $0 \sim n-1$），則中間木樁的索引滿足：
$$0 < i < n - 1$$
也就是說，$i$ 的最小可能值是 `1`，最大可能值是 `n - 2`。

在這個範圍內，任何一個位置 $i$ 都擁有完完整整的兩個鄰居：
- **左鄰居**：位於位置 $i - 1$，高度為 `h[i - 1]`
- **右鄰居**：位於位置 $i + 1$，高度為 `h[i + 1]`

#### 中間損壞處的修復公式：
只要判定 `h[i] == 0`，這根木樁的修復費用就是：
$$\text{cost} = \min(h[i - 1], h[i + 1])$$

以範例一 `h = [3, 0, 5, 0, 4]` 為例：
- $i = 1$ 時：$h[1] == 0$，左邊 $h[0]=3$，右邊 $h[2]=5$，花費 $\min(3, 5) = 3$。
- $i = 3$ 時：$h[3] == 0$，左邊 $h[2]=5$，右邊 $h[4]=4$，花費 $\min(5, 4) = 4$。
- 兩者相加即為 $3 + 4 = 7$！

只要位置在內部，這條公式無懈可擊。但千萬別高興得太早——最危險的考驗正在兩端等著我們！

In [ ]:
# ==========================================
# [2] Code 範例區：計算所有內部破損處的修復花費
# ==========================================

h = [3, 0, 5, 0, 4]
n = len(h)

inner_total = 0
# 僅遍歷內部位置 1 到 n-2
for i in range(1, n - 1):
    if h[i] == 0:
        cost = min(h[i-1], h[i+1])
        inner_total += cost
        print(f"內部位置 {i} 破損 ➔ 左={h[i-1]}, 右={h[i+1]} ➔ 修復花費={cost}")

print(f"內部破損總修復花費: {inner_total}")


In [ ]:
# ==========================================
# [3] Code 填空題：補齊內部相鄰索引
# 任務說明：請將下方 ___ 替換，正確讀取 i 的左鄰居與右鄰居。
# ==========================================

h = [10, 20, 0, 30, 0, 15]
i = 2  # h[2] 為 0

left_neighbor = h[___]   # i 的左邊
right_neighbor = h[___]  # i 的右邊
cost = min(left_neighbor, right_neighbor)

print(f"位置 {i} 修復花費: {cost}")
# 預期輸出：位置 2 修復花費: 20


In [ ]:
# ==========================================
# [4] Code 練習題：計算純內部破損序列的總花費
#
# 【公開測試資料 1】
# 輸入：[3, 0, 5, 0, 4]（兩端完好）
# 預期輸出：7
#
# 【公開測試資料 2】
# 輸入：[10, 20, 0, 30, 0, 15]（兩端完好）
# 預期輸出：35
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def solve_inner_only(h):
    ans = 0
    for i in range(1, len(h) - 1):
        if h[i] == 0:
            ans += min(h[i-1], h[i+1])
    return ans

# 執行測試
print("測試 1:", solve_inner_only([3, 0, 5, 0, 4]))
print("測試 2:", solve_inner_only([10, 20, 0, 30, 0, 15]))


In [ ]:
# ==========================================
# [5] Code 挑戰題：統計內部破損修復時，有多少次是「取左邊」而非「取右邊」
# 任務說明：請寫迴圈統計，在所有內部破損處中，
#          左鄰居小於右鄰居（h[i-1] < h[i+1]）的次數共有幾次？
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


### 14.9.4 🚫 邊界越界大陷阱：最左端（`i == 0`）與最右端（`i == n-1`）

現在，我們來揭開這道題目最殘酷的陷阱——**邊界越界**！

很多初學者寫程式時，直接寫出這樣的迴圈：
```python
for i in range(n):
    if h[i] == 0:
        total += min(h[i-1], h[i+1])  # 💣 致命地雷！
```
如果在內部跑，完全沒問題；但只要**第一根木樁（$i = 0$）或最後一根木樁（$i = n - 1$）**斷掉了，會發生什麼慘劇？

#### 💣 慘劇一：當 $i = n - 1$（最右端損壞）
此時程式會嘗試存取 `h[i + 1]`，也就是 `h[n]`！
因為串列最大索引只有 `n - 1`，Python 直譯器會直接無情噴出：
**`IndexError: list index out of range`**！程式直接當場中斷死亡！

#### 💣 慘劇二：當 $i = 0$（最左端損壞，更隱蔽的幽靈錯誤！）
此時程式嘗試存取 `h[i - 1]`，也就是 `h[-1]`！
在 Python 語法中，**負數索引代表從倒數第一個開始數**！
- `h[-1]` 拿到的不是左鄰居，而是整排圍籬**最右邊的最後一根木樁**！
- 拿農場最東邊的木樁，來當作最西邊木樁的鄰居，邏輯徹底荒腔走板！
- 程式**不會報錯**，但算出來的數字完全錯誤，拿到痛心的 `WA`（Wrong Answer）！

#### 🛡️ 正確的邊界規則：
- **最左端（$i = 0$）**：沒有左鄰居，修復費用**直接等於右鄰居 `h[1]`**！
- **最右端（$i = n - 1$）**：沒有右鄰居，修復費用**直接等於左鄰居 `h[n - 2]`**！

這就是為什麼邊界必須被特殊保護！

In [ ]:
# ==========================================
# [2] Code 範例區：最左端與最右端破損的正確防禦提取
# ==========================================

# 官方範例二：h = [0, 6, 8, 0]
h = [0, 6, 8, 0]
n = len(h)

print("=== 兩端邊界防禦範例 ===")
# 檢驗最左端 (i = 0)
if h[0] == 0:
    cost_left = h[1]  # 只能看右邊 h[1]
    print(f"最左端 i=0 破損 ➔ 取右鄰居 h[1]={h[1]} ➔ 修復花費={cost_left}")

# 檢驗最右端 (i = n - 1)
if h[n - 1] == 0:
    cost_right = h[n - 2]  # 只能看左邊 h[n - 2]
    print(f"最右端 i={n-1} 破損 ➔ 取左鄰居 h[{n-2}]={h[n-2]} ➔ 修復花費={cost_right}")

print(f"範例二總花費: {cost_left + cost_right}（預期 14）")


In [ ]:
# ==========================================
# [3] Code 填空題：補齊邊界索引防禦
# 任務說明：請將下方 ___ 填入正確索引，防禦最左與最右端。
# ==========================================

h = [0, 15, 20, 0]
n = len(h)

# 最左端破損，取右鄰居
cost_0 = h[___]

# 最右端破損，取左鄰居
cost_last = h[___]

print(f"左端花費: {cost_0}, 右端花費: {cost_last}")
# 預期輸出：左端花費: 15, 右端花費: 20


In [ ]:
# ==========================================
# [4] Code 練習題：專門處理兩端破損的檢驗函式
#
# 【公開測試資料 1】
# 輸入：[0, 12, 18, 0]
# 預期輸出：左端 12，右端 18
#
# 【公開測試資料 2】
# 輸入：[5, 12, 18, 0]
# 預期輸出：左端無破損，右端 18
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def check_endpoints(h):
    n = len(h)
    left_msg = f"左端 {h[1]}" if h[0] == 0 else "左端無破損"
    right_msg = f"右端 {h[n-2]}" if h[n-1] == 0 else "右端無破損"
    return f"{left_msg}，{right_msg}"

# 執行測試
print("測試 1:", check_endpoints([0, 12, 18, 0]))
print("測試 2:", check_endpoints([5, 12, 18, 0]))


In [ ]:
# ==========================================
# [5] Code 挑戰題：長度為 2 的極小圍籬特例
# 任務說明：當 n = 2 時，例如 [0, 7] 或 [7, 0]。
#          請撰寫一段代碼，驗證其是否能正確計算破損端的費用為 7？
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


### 14.9.5 🥋 傳統邊界特判解法：三分支互斥防禦與總花費累加

把前兩小節的內部公式與兩端邊界規則組合在一起，我們就得到了考場上最經典、最直覺的**「三分支條件特判法」**！

#### 三分支防禦心法：
在遍歷 `for i in range(n):` 的過程中，當遇到 `h[i] == 0` 時，我們依據**位置索引 $i$** 進行三路分流：
1. **分支一（最左端）**：`if i == 0:`
   - 此時只有右鄰居，花費為 `h[1]`。
2. **分支二（最右端）**：`elif i == n - 1:`
   - 此時只有左鄰居，花費為 `h[n - 2]`。
3. **分支三（中間一般位置）**：`else:`
   - 兩側都有鄰居，花費為 `min(h[i - 1], h[i + 1])`。

```python
total_cost = 0
for i in range(n):
    if h[i] == 0:
        if i == 0:
            total_cost += h[1]
        elif i == n - 1:
            total_cost += h[n - 2]
        else:
            total_cost += min(h[i - 1], h[i + 1])
```

#### 這套寫法的優點：
- **完全符合自然直覺**：題目怎麼規定，程式碼就怎麼寫，思路毫無繞彎。
- **嚴格防禦例外**：絕不可能出現 `h[-1]` 倒轉或 `h[n]` 越界，100% 安全穩健！
- 任何初學者只要寫出這套邏輯，就能在 APCS 考場穩拿 100 分滿分！

In [ ]:
# ==========================================
# [2] Code 範例區：三分支特判演算法完整示範
# ==========================================

def solve_fence_branching(n, h):
    total_cost = 0
    for i in range(n):
        if h[i] == 0:
            if i == 0:
                total_cost += h[1]
            elif i == n - 1:
                total_cost += h[n - 2]
            else:
                total_cost += min(h[i - 1], h[i + 1])
    return total_cost

# 驗證官方範例一與範例二
print("範例一結果:", solve_fence_branching(5, [3, 0, 5, 0, 4]))  # 預期 7
print("範例二結果:", solve_fence_branching(4, [0, 6, 8, 0]))        # 預期 14


In [ ]:
# ==========================================
# [3] Code 填空題：補齊三分支分流條件
# 任務說明：請將下方 ___ 替換為正確的索引判斷條件。
# ==========================================

n = 5
h = [0, 10, 0, 20, 0]
total = 0

for i in range(n):
    if h[i] == 0:
        if i == ___:            # 最左端
            total += h[1]
        elif i == ___:        # 最右端
            total += h[n - 2]
        else:
            total += min(h[i-1], h[i+1])

print(f"總修復花費: {total}")
# 預期輸出：總修復花費: 40 (10 + 10 + 20)


In [ ]:
# ==========================================
# [4] Code 練習題：測試綜合破損序列
#
# 【公開測試資料 1】
# 輸入：n=6, h=[10, 20, 0, 30, 0, 15]
# 預期輸出：35
#
# 【公開測試資料 2】
# 輸入：n=4, h=[0, 100, 50, 0]
# 預期輸出：150
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def test_branching(n, h):
    cost = 0
    for i in range(n):
        if h[i] == 0:
            if i == 0:
                cost += h[1]
            elif i == n - 1:
                cost += h[n - 2]
            else:
                cost += min(h[i-1], h[i+1])
    return cost

# 執行測試
print("測試 1:", test_branching(6, [10, 20, 0, 30, 0, 15]))
print("測試 2:", test_branching(4, [0, 100, 50, 0]))


In [ ]:
# ==========================================
# [5] Code 挑戰題：印出每次修補的明細日誌
# 任務說明：在迴圈累加時，格式化印出每次修復的木樁編號、
#          採用的分支類別（左端/右端/中間）與本次花費。
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


### 14.9.6 ✨ 陣列降維神技：哨兵前後加框法（Sentinel Technique）

三分支特判雖然直觀，但每次寫 `if i == 0` 和 `elif i == n-1` 難免覺得程式碼有點囉嗦。
在進階演算法與競技程式設計中，有一種被稱為**「哨兵（Sentinel）加框法」**的絕妙神技，能讓我們**徹底消滅所有邊界特判**！

#### 什麼是「哨兵加框」？
既然最左端和最右端是因為「缺了一個鄰居」才需要特判，那我們何不**主動在圍籬的最前面和最後面，各放一根『虛擬無敵大木樁』**呢？

題目告訴我們，每根木樁的高度最高只有 100。
我們在串列的最前端補一個 $105$（或無限大 $\infty$），在最後端也補一個 $105$：
```python
padded_h = [105] + h + [105]
```

```text
   原序列:              [0,     6,    8,     0]
   加框後:      [105]   [0,     6,    8,     0]   [105]
   新索引:        0      1      2     3      4      5
                       ▲                    ▲
                  原最左端(新1)         原最右端(新4)
```

#### 奇蹟發生了！
讓我們來看看加框後的原最左端（新索引 1，高度為 0）：
- 它的左鄰居變成了哨兵 $105$！
- 它的右鄰居是原來的 $6$！
- 套用通用公式：$\min(105, 6) = 6$！**居然自動選中了右鄰居 6**！

再來看看原最右端（新索引 4，高度為 0）：
- 它的左鄰居是原來的 $8$！
- 它的右鄰居變成了哨兵 $105$！
- 套用通用公式：$\min(8, 105) = 8$！**居然自動選中了左鄰居 8**！

因為哨兵比任何真實木樁都要高，`min()` 永遠不會選中哨兵！
原本棘手的邊界特例，一瞬間被降維成了普通的內部位置！整段程式碼只剩下乾淨俐落的一行迴圈！

In [ ]:
# ==========================================
# [2] Code 範例區：哨兵前後加框法震撼示範
# ==========================================

def solve_fence_sentinel(n, h):
    # 前後各補一個極大值 105 作為哨兵守衛
    padded = [105] + h + [105]
    total_cost = 0
    
    # 遍歷原序列在加框後的對應索引（從 1 到 n）
    for i in range(1, n + 1):
        if padded[i] == 0:
            total_cost += min(padded[i-1], padded[i+1])
            
    return total_cost

# 驗證包含邊界破損的官方範例二
print("=== 哨兵加框法測試 ===")
print("範例一 [3, 0, 5, 0, 4] ➔ 結果:", solve_fence_sentinel(5, [3, 0, 5, 0, 4])) # 7
print("範例二 [0, 6, 8, 0]       ➔ 結果:", solve_fence_sentinel(4, [0, 6, 8, 0]))       # 14


In [ ]:
# ==========================================
# [3] Code 填空題：補齊加框串列與哨兵常數
# 任務說明：請將下方 ___ 替換，在串列前後各加上包含哨兵的單元素串列。
# ==========================================

raw_h = [0, 20, 30, 0]
INF = 999  # 足夠大的哨兵常數

# 在 raw_h 前後拼接 [INF]
padded = [___] + raw_h + [___]

print("加框後的長度:", len(padded))
# 預期輸出：加框後的長度: 6


In [ ]:
# ==========================================
# [4] Code 練習題：使用哨兵加框法重構解題函式
#
# 【公開測試資料 1】
# 輸入：n=5, h=[0, 10, 0, 20, 0]
# 預期輸出：40
#
# 【公開測試資料 2】
# 輸入：n=3, h=[5, 8, 6]
# 預期輸出：0
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def solve_sentinel_test(n, h):
    p = [999] + h + [999]
    ans = 0
    for i in range(1, n + 1):
        if p[i] == 0:
            ans += min(p[i-1], p[i+1])
    return ans

# 執行測試
print("測試 1:", solve_sentinel_test(5, [0, 10, 0, 20, 0]))
print("測試 2:", solve_sentinel_test(3, [5, 8, 6]))


In [ ]:
# ==========================================
# [5] Code 挑戰題：使用 float('inf') 作為無窮大哨兵
# 任務說明：在 Python 中，float('inf') 代表正無限大，任何數字都小於它。
#          請嘗試使用 [float('inf')] + h + [float('inf')] 實作哨兵法！
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


### 14.9.7 🏆 完整 AC 模組拼裝與雙解法（傳統特判 vs 哨兵加框）對照

學習寫程式最美妙的時刻，就是**同一道問題能夠在心中擁有兩種截然不同、卻殊途同歸的解法**！

讓我們把兩種解法並列在一起進行深度對照：

| 比較維度 | 策略一：傳統三分支條件特判法 | 策略二：哨兵加框法（Sentinel Technique） |
| :--- | :--- | :--- |
| **核心思維** | 依據索引位置分流：左端、右端、中間 | 在序列前後補上比 100 更大的虛擬木樁 |
| **程式行數** | 約 12～15 行 | **僅約 6～7 行** |
| **邏輯分支** | 3 個 `if-elif-else` 判斷 | **0 分支**，統一套用 `min(p[i-1], p[i+1])` |
| **初學者感知** | 直觀易懂，完全對應中文題意 | 巧奪天工，展現高階競賽選手的優雅思維 |
| **記憶體開銷** | $O(1)$，完全不需要新串列 | $O(n)$，產生一個長度加 2 的新串列 |

在 APCS 考場上：
- 若你求**穩妥不繞彎**，寫「策略一」絕對能拿滿分；
- 若你追求**代碼簡約美與打字速度**，寫「策略二」能在 1 分鐘內極速敲完，而且絕不會寫錯索引！

兩種方法都是優秀程式設計師工具箱中的利刃！

In [ ]:
# ==========================================
# [2] Code 範例區：雙解法對照與正確性一致性檢驗
# ==========================================

# 策略一：傳統三分支特判法
def solve_v1(n, h):
    ans = 0
    for i in range(n):
        if h[i] == 0:
            if i == 0:
                ans += h[1]
            elif i == n - 1:
                ans += h[n - 2]
            else:
                ans += min(h[i-1], h[i+1])
    return ans

# 策略二：哨兵加框極速法
def solve_v2(n, h):
    p = [105] + h + [105]
    return sum(min(p[i-1], p[i+1]) for i in range(1, n+1) if p[i] == 0)

# 交叉比對四組官方範例
samples = [
    (5, [3, 0, 5, 0, 4]),
    (4, [0, 6, 8, 0]),
    (6, [10, 20, 0, 30, 0, 15]),
    (3, [5, 8, 6])
]

print("=== 雙解法交叉驗證報告 ===")
for i, (n, h) in enumerate(samples, 1):
    r1 = solve_v1(n, h)
    r2 = solve_v2(n, h)
    print(f"範例 {i}: 特判法={r1}, 哨兵法={r2} ➔ {'一致 PASS' if r1 == r2 else 'FAIL'}")


In [ ]:
# ==========================================
# [3] Code 填空題：補齊一行列表生成式加總
# 任務說明：請將下方 ___ 替換，利用 sum() 與生成式完成花費加總。
# ==========================================

p = [105, 3, 0, 5, 0, 4, 105]
n = 5

# 生成式加總所有 p[i] == 0 的 min 鄰居花費
total = ___(min(p[i-1], p[i+1]) for i in range(1, n+1) if p[i] == 0)

print(f"總花費: {total}")
# 預期輸出：總花費: 7


In [ ]:
# ==========================================
# [4] Code 練習題：完整的考場 I/O 模擬執行
#
# 【公開測試資料 1】
# 輸入字串："4\n0 6 8 0"
# 預期輸出：14
#
# 【公開測試資料 2】
# 輸入字串："5\n3 0 5 0 4"
# 預期輸出：7
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def run_mock_io(input_str):
    lines = input_str.strip().split("\n")
    n = int(lines[0])
    h = list(map(int, lines[1].split()))
    p = [105] + h + [105]
    return sum(min(p[i-1], p[i+1]) for i in range(1, n+1) if p[i] == 0)

# 執行測試
print("測試 1:", run_mock_io("4\n0 6 8 0"))
print("測試 2:", run_mock_io("5\n3 0 5 0 4"))


In [ ]:
# ==========================================
# [5] Code 挑戰題：記錄被修補後木樁序列的新樣貌
# 任務說明：請撰寫程式，修復後不僅計算總花費，
#          同時輸出整排木樁被修復後的最終高度串列！
#          例如 [3, 0, 5, 0, 4] 修復後應變成 [3, 3, 5, 4, 4]。
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


### 14.9.8 🧪 極端邊界壓力測試、無損壞檢驗與考場常見失分地雷排查

在送出解答奪取滿分之前，我們必須針對所有可能讓程式報錯崩潰的**極端測資邊界**進行全面壓力測試！

#### 💣 考場 4 大致命 WA/RE 地雷排查：
1. **地雷一：兩端皆破損（雙重邊界夾擊）**：
   - 例如序列 `[0, 6, 8, 0]`，第 0 格和最後一格同時是 0。程式必須同時正確防禦左端與右端，不可顧此失彼。
2. **地雷二：完全無破損（純完好圍籬）**：
   - 例如 `[5, 8, 6]`，序列中完全沒有 0。程式必須安全跳過所有累加，正確輸出 `0`，不可發生未定義變數錯誤。
3. **地雷三：極小規模圍籬（$n = 2$）**：
   - 題目範圍 $2 \le n \le 100$。當 $n = 2$ 時，例如 `[0, 10]`：
     - 只有兩根木樁，左邊是 0，右鄰居是 10，答案為 10。
     - 很多同學寫特判時誤設 `range(1, n-1)`，在 $n=2$ 時中間範圍直接變成空區間，此時最右端 $i=n-1=1$ 必須能正確識別！
4. **地雷四：哨兵數值設定過小**：
   - 題目說明木樁高度最大可達 $100$。如果你的哨兵設為 $10$ 或 $99$，遇到高達 $100$ 的木樁時，`min(100, 哨兵)` 就會把哨兵選進去，導致算出的花費嚴重偏低！因此哨兵必須**嚴格大於 100**（例如 105 或 999）！

#### ⏱️ 時間與空間複雜度分析：
- **時間複雜度**：$O(n)$。僅需線性走訪一次長度為 $n$ 的串列，即使 $n = 100$，運算次數僅幾百次，在 0.001 秒內瞬間通過，時間複雜度最佳！
- **空間複雜度**：$O(1)$（特判法）或 $O(n)$（加框法）。佔用記憶體小於 1KB，遠低於 64MB 限制。

In [ ]:
# ==========================================
# [2] Code 範例區：全極端邊界測試套件執行
# ==========================================

def solve(n, h):
    p = [105] + h + [105]
    return sum(min(p[i-1], p[i+1]) for i in range(1, n+1) if p[i] == 0)

# 構造極端邊界測試套件
extreme_suite = [
    ("極小規模 n=2 (左端破損)", 2, [0, 10], 10),
    ("極小規模 n=2 (右端破損)", 2, [10, 0], 10),
    ("兩端同時破損 [0, 6, 8, 0]", 4, [0, 6, 8, 0], 14),
    ("完全無破損 [5, 8, 6]", 3, [5, 8, 6], 0),
    ("最高高度 100 邊界測試", 3, [100, 0, 100], 100),
    ("間隔交替損壞 [0, 5, 0, 5, 0]", 5, [0, 5, 0, 5, 0], 15)
]

print("=== 極端邊界壓力測試報告 ===")
for name, n, h, exp in extreme_suite:
    act = solve(n, h)
    status = "PASS" if act == exp else "FAIL"
    print(f"[{status}] {name:30s} ➔ 預期: {exp:3d}, 實際: {act:3d}")


In [ ]:
# ==========================================
# [3] Code 填空題：安全哨兵門檻設定
# 任務說明：木樁最大高度為 100，請將 ___ 替換為安全且大於 100 的哨兵數值。
# ==========================================

# 只要大於 100 即可
SAFE_SENTINEL = ___

print("哨兵安全嗎？", SAFE_SENTINEL > 100)
# 預期輸出：哨兵安全嗎？ True


In [ ]:
# ==========================================
# [4] Code 練習題：測試連續多組隨機邊界
#
# 【公開測試資料 1】
# 輸入：n=5, h=[0, 5, 0, 5, 0]
# 預期輸出：15
#
# 【公開測試資料 2】
# 輸入：n=3, h=[100, 0, 100]
# 預期輸出：100
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

def test_extreme(n, h):
    return solve(n, h)

# 執行測試
print("測試 1:", test_extreme(5, [0, 5, 0, 5, 0]))
print("測試 2:", test_extreme(3, [100, 0, 100]))


In [ ]:
# ==========================================
# [5] Code 挑戰題：自製一個隨機無連續破損圍籬產生器
# 任務說明：請寫一個小函式，給定長度 n，隨機產生一排合法的圍籬木樁序列
#          （高度介於 0~100，且保證絕對不出現連續兩個 0）。
# （本題為自由挑戰題，無公開測資，請自主思考設計並測試驗證）
# ==========================================
# 請在下方撰寫你的程式碼：


## 🏆 恭喜通關！單元學習總結與榮耀通關徽章

🎉 **太棒了！你已經成功攻克了 APCS 經典實作真題——g595. 修補圍籬！**

讓我們回顧一下在這一單元中，你所解鎖的 5 大核心陣列演算法超能力：
1. 🧱 **一維陣列相鄰走訪探測**：熟練掌握 `h[i-1]` 與 `h[i+1]` 的相鄰比對與資料提取。
2. 🚫 **邊界越界例外防禦**：深刻理解最左端 `i=0` 與最右端 `i=n-1` 的存取邊界，徹底告別 `IndexError` 與負索引倒轉致命地雷。
3. 🥋 **經典三分支分流特判**：掌握 `if i == 0 ... elif i == n-1 ... else` 的穩健防禦思維。
4. ✨ **陣列降維神技「哨兵加框法」**：前後各補虛擬無敵守衛（105），以極致優雅的 6 行代碼消滅所有邊界分支！
5. 🧪 **全邊界極端壓力測試**：完整驗證兩端破損、無損壞、極小規模 $n=2$ 等各種競賽測資情境。

---

### 🎖️ APCS 陣列相鄰探測與哨兵神技・通關徽章

```text
  +=========================================================+
  |                                                         |
  |       🏆 APCS 實作初級真題特訓：修補圍籬通關 🏆          |
  |                                                         |
  |   題目名稱：g595. 修補圍籬 (Repair Fence)               |
  |   核心考點：一維陣列相鄰探測、邊界條件防禦、哨兵加框神技  |
  |   解題品質：8 階梯極致緩坡超詳解 (48 Cells)             |
  |                                                         |
  |   ✅ 內部相鄰取較小值 min(h[i-1], h[i+1])               |
  |   ✅ 兩端邊界防禦：左端 h[1]、右端 h[n-2]               |
  |   ✅ 哨兵前後加框法 [105] + h + [105] 極速秒殺           |
  |   ✅ 雙平台滿分通關解答庫配備完畢                       |
  |                                                         |
  |   能力評級：APCS 陣列演算法與邊界控制 100% 掌握！       |
  |                                                         |
  +=========================================================+
```

陣列邊界的難關已經被你徹底征服！帶著這份扎實的自信，繼續迎接下一座榮耀高峰！

## 💻【附錄：雙平台滿分通關解答庫】

在 APCS 考場與線上刷題平台（如 ZeroJudge）之間，輸入測資的提供方式存在關鍵差異：

| 評判維度 | 🥇 APCS 官方實作正式考場版 | 🌐 ZeroJudge 線上評判萬用 AC 版 |
| :--- | :--- | :--- |
| **輸入機制** | **保證第一行給 $n$，第二行給 $n$ 個高度** | **整批多筆測資連續灌入（以 EOF 結尾）** |
| **讀取方式** | 標準 2 次 `input()` 即可完成輸入 | 必須使用 `sys.stdin.read().split()` 串流讀取 |
| **程式碼長度** | 約 10～12 行，極致簡短、考場不易打錯 | 結構化模組封裝，支援連續多組測資自動解析 |
| **Colab 執行體驗** | 點擊執行需手動貼上 2 行測資 | 內建官方範例自動化測試驅動器，一鍵全綠燈 |

下方為大家分別提供兩種版本的完整滿分程式碼！

### 📝 版本一：APCS 官方實作考場專用版（極簡 12 行考場滿分版）

> 📌 **考場攻略**：  
> 在 APCS 正式考場中，題目保證輸入只有兩行：第一行是木樁數 $n$，第二行是 $n$ 個木樁高度。
> 我們使用「哨兵加框法」，在串列前後各補一個 105，直接一行生成式算完總和，12 行代碼極速 AC！
> 
> 💡 **在 Colab 中執行時**：點擊下方執行按鈕，請在輸入框中貼上測試資料（共兩行）即可看見輸出。

In [ ]:
# =====================================================================
# 📝 版本一：APCS 官方實作考場專用版（極簡 12 行哨兵滿分版）
# =====================================================================

# 1. 讀取輸入資料（考場保證剛好 2 行）
n = int(input())                          # 第一行：木樁總數 n
h = list(map(int, input().split()))       # 第二行：n 根木樁的高度序列

# 2. 前後各補一個哨兵 105（大於所有木樁高度上限 100）
p = [105] + h + [105]

# 3. 遍歷原圍籬位置，累加所有損壞處的修復花費
total_cost = 0
for i in range(1, n + 1):
    if p[i] == 0:
        total_cost += min(p[i-1], p[i+1]) # 哨兵保證邊界永遠自動取真實鄰居

# 4. 輸出總花費
print(total_cost)


### 🌐 版本二：ZeroJudge 線上評判萬用 AC 版（多測資 EOF 處理 + 內建測試器）

> 📌 **線上評判攻略**：  
> 在 ZeroJudge 等線上 OJ 系統中，評測機通常會將多筆測試資料連續灌入（以 EOF 結尾）。
> 本版本使用 `sys.stdin.read().split()` 將所有數值一次讀入，使用指針循序解析，完美避開 `EOFError`！
> 
> 💡 **在 Colab 中執行時**：程式內建全自動化測試驅動器，自動執行 4 組官方範例測資，全數通過將印出綠色打勾標記！

In [ ]:
# =====================================================================
# 🌐 版本二：ZeroJudge 線上評判萬用 AC 版（含 Colab 自動測試驅動器）
# =====================================================================
import sys

def solve_fence(n, h):
    """
    修補圍籬核心演算法（哨兵加框法）
    :param n: 木樁總數
    :param h: 木樁高度序列
    :return: 總修復花費
    """
    p = [105] + h + [105]
    return sum(min(p[i-1], p[i+1]) for i in range(1, n+1) if p[i] == 0)

# ---------------------------------------------------------------------
# 【ZeroJudge 官方提交程式碼範本】
# 若要在 ZeroJudge 提交，請複製下方函式內容至解題系統：
# def main():
#     tokens = sys.stdin.read().split()
#     if not tokens: return
#     idx = 0
#     while idx < len(tokens):
#         n = int(tokens[idx])
#         idx += 1
#         h = [int(x) for x in tokens[idx:idx+n]]
#         idx += n
#         print(solve_fence(n, h))
# ---------------------------------------------------------------------

# =====================================================================
# 🧪 Colab 本地自動化測試檢驗套件（點擊播放鍵自動執行）
# =====================================================================
test_cases = [
    {
        "name": "官方範例一 (內部兩處損壞)",
        "n": 5, "h": [3, 0, 5, 0, 4],
        "expected": 7
    },
    {
        "name": "官方範例二 (兩端同時損壞)",
        "n": 4, "h": [0, 6, 8, 0],
        "expected": 14
    },
    {
        "name": "官方範例三 (多處交替損壞)",
        "n": 6, "h": [10, 20, 0, 30, 0, 15],
        "expected": 35
    },
    {
        "name": "官方範例四 (完全無損壞)",
        "n": 3, "h": [5, 8, 6],
        "expected": 0
    }
]

print("=== g595. 修補圍籬 全自動化測試報告 ===")
all_passed = True
for tc in test_cases:
    actual = solve_fence(tc["n"], tc["h"])
    passed = (actual == tc["expected"])
    status = "✅ PASS" if passed else "❌ FAIL"
    if not passed:
        all_passed = False
    print(f"{status} | {tc['name']} ➔ 預期: {tc['expected']}, 實際: {actual}")

if all_passed:
    print("\n🎉 恭喜！所有官方範例測試全數通過！可安心提交至 APCS 考場與 ZeroJudge！")
else:
    print("\n⚠️ 有測試資料未通過，請檢查邊界或哨兵加框邏輯！")
